In [ ]:
# This notebook runs our framework on Knapsack problems.


## TODO: Fix the current circuit implementation as it is meant only for Maxcut. 

In [1]:
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
import random

from qiskit import transpile
from qiskit.circuit import Parameter,ParameterExpression
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeManilaV2,FakeNairobiV2,FakeMelbourneV2,FakeGuadalupeV2

import sys
sys.path.append("../")
from clapton.clapton import claptonize
from clapton.circuit_manipulation import transform_to_allowed_gates,qiskit_to_stim, modify_circuit, multi_angle_qaoa_circuit
from graphs_gen import generate_random_complete_graph,generate_k_regular_graph

In [2]:
from qiskit_optimization.applications import Knapsack
from qiskit_optimization.converters import QuadraticProgramToQubo

prob = Knapsack(values=[1, 2, 3, 4, 5, 6], weights=[1, 2, 3, 4, 5,6], max_weight=10)
qp = prob.to_quadratic_program()
print(qp.prettyprint())

# intermediate QUBO form of the optimization problem
conv = QuadraticProgramToQubo()
qubo = conv.convert(qp)

# qubit Hamiltonian and offset
op, offset = qubo.to_ising()
print(f"num qubits: {op.num_qubits}, offset: {offset}\n")
print(op)

cost_hamiltonian = op

Problem name: Knapsack

Maximize
  x_0 + 2*x_1 + 3*x_2 + 4*x_3 + 5*x_4 + 6*x_5

Subject to
  Linear constraints (1)
    x_0 + 2*x_1 + 3*x_2 + 4*x_3 + 5*x_4 + 6*x_5 <= 10  'c0'

  Binary variables (6)
    x_0 x_1 x_2 x_3 x_4 x_5

num qubits: 10, offset: 1320.5

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII', 'IIIIIIIIZZ', 'IIIIIIIZIZ', 'IIIIIIZIIZ', 'IIIIIZIIIZ', 'IIIIZIIIIZ', 'IIIZIIIIIZ', 'IIZIIIIIIZ', 'IZIIIIIIIZ', 'ZIIIIIIIIZ', 'IIIIIIIZZI', 'IIIIIIZIZI', 'IIIIIZIIZI', 'IIIIZIIIZI', 'IIIZIIIIZI', 'IIZIIIIIZI', 'IZIIIIIIZI', 'ZIIIIIIIZI', 'IIIIIIZZII', 'IIIIIZIZII', 'IIIIZIIZII', 'IIIZIIIZII', 'IIZIIIIZII', 'IZIIIIIZII', 'ZIIIIIIZII', 'IIIIIZZIII', 'IIIIZIZIII', 'IIIZIIZIII', 'IIZIIIZIII', 'IZIIIIZIII', 'ZIIIIIZIII', 'IIIIZZIIII', 'IIIZIZIIII', 'IIZIIZIIII', 'IZIIIZIIII', 'ZIIIIZIIII', 'IIIZZIIIII', 'IIZIZIIIII', 'IZIIZIIIII', 'ZIIIZIIIII', 'IIZZIIIIII', 'IZIZIIIIII', 'ZIIZIIIII

In [3]:
from qiskit.circuit.library import QAOAAnsatz

circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=1)

dec_circ = circuit.decompose().decompose().decompose()


In [4]:
dag = circuit_to_dag(dec_circ)
param_list = [list(node.op.params[0].parameters)[0].name for node in dag.op_nodes() if node.op.params and isinstance(node.op.params[0], ParameterExpression)]

In [5]:
paulis,coeffs = cost_hamiltonian.paulis.to_labels(),cost_hamiltonian.coeffs.real
reversed_paulis = [p[::-1] for p in paulis]

In [6]:
assert len(paulis)+len(paulis[0])==len(param_list)

In [7]:
gamma_counter,beta_counter = 0,0
for node in dag.op_nodes():
    if node.op.params and isinstance(node.op.params[0], ParameterExpression):
        # phi = Parameter(f'phi_{i}')

        param_name = list(node.op.params[0].parameters)[0].name 
        if "β" in param_name:
            beta = Parameter(f'beta_{beta_counter}')
            new_params = [beta if (isinstance(p, ParameterExpression)) else p for p in node.op.params]
            new_op = node.op.copy()
            new_op.params = new_params  # Create a modified version of the operation
            dag.substitute_node(node, new_op)  # Replace the node in the DAG
            beta_counter+=1
        elif "γ" in param_name:
            gamma = Parameter(f'gamma_{gamma_counter}')
            new_params = [gamma if (isinstance(p, ParameterExpression)) else p for p in node.op.params]
            new_op = node.op.copy()
            new_op.params = new_params  # Create a modified version of the operation
            dag.substitute_node(node, new_op)  # Replace the node in the DAG
            gamma_counter+=1
# Convert DAG back to a circuit
new_qc = dag_to_circuit(dag)
new_dag = circuit_to_dag(new_qc)
circuit = new_qc
 

/var/folders/w0/hhmpfx790159l9v9s3ysf69c0000gn/T/ipykernel_58312/3517541982.py:22: DeprecationWarning: The property ``qiskit.dagcircuit.dagcircuit.DAGCircuit.duration`` is deprecated as of qiskit 1.3.0. It will be removed in Qiskit 2.0.0.
  new_qc = dag_to_circuit(dag)
/var/folders/w0/hhmpfx790159l9v9s3ysf69c0000gn/T/ipykernel_58312/3517541982.py:22: DeprecationWarning: The property ``qiskit.dagcircuit.dagcircuit.DAGCircuit.unit`` is deprecated as of qiskit 1.3.0. It will be removed in Qiskit 2.0.0.
  new_qc = dag_to_circuit(dag)


In [8]:
# Transform qiskit circ. to stim.
modified_circ = modify_circuit(circuit)
pcirc = transform_to_allowed_gates(modified_circ)

stim_circ = qiskit_to_stim(pcirc)

stim_circ.stim_circuit().diagram()

q0: -S-SQRT_X-Z-I------------@---@------------@---@------------------@---@------------------------@---@------------------------------@---@------------------------------------@---@------------------------------------------@---@------------------------------------------------@---@------------------------------------------------------@---@-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
                             |   |            |   |                  |   |                        |   |                              |   |                                    |   |                                          |   |                                                |   |                                                      |   |
q1: ------------S-SQRT_X-S-I-X-I-X------------|---|-@---@------------|---|-@---@------------------|---|-@---@------------------------|---|-@---@------------------------------|---|-@---@------------------------------------|---|-@---@------------------------------------------|---|-@---@------------------------------------------------|---|-----------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
                                              |   | |   |            |   | |   |                  |   | |   |                        |   | |   |                              |   | |   |                                    |   | |   |                                          |   | |   |                                                |   |                 |   |
q2: -----------------------------S-SQRT_X-S-I-X-I-X-X-I-X------------|---|-|---|-@---@------------|---|-|---|-@---@------------------|---|-|---|-@---@------------------------|---|-|---|-@---@------------------------------|---|-|---|-@---@------------------------------------|---|-|---|-@---@------------------------------------------|---|-----------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                     |   | |   | |   |            |   | |   | |   |                  |   | |   | |   |                        |   | |   | |   |                              |   | |   | |   |                                    |   | |   | |   |                                          |   |                 |   |                   |   |
q3: ----------------------------------------------------S-SQRT_X-S-I-X-I-X-X-I-X-X-I-X------------|---|-|---|-|---|-@---@------------|---|-|---|-|---|-@---@------------------|---|-|---|-|---|-@---@------------------------|---|-|---|-|---|-@---@------------------------------|---|-|---|-|---|-@---@------------------------------------|---|-----------------|---|-------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------------------------------
                                                                                                  |   | |   | |   | |   |            |   | |   | |   | |   |                  |   | |   | |   | |   |                        |   | |   | |   | |   |                              |   | |   | |   | |   |                                    |   |                 |   |                   |   |                   |   |
q4: ---------------------------------------------------------------------------------S-SQRT_X-S-I-X-I-X-X-I-X-X-I-X-X-I-X------------|---|-|---|-|---|-|---|-@---@------------|---|-|---|-|---|-|---|-@---@------------------|---|-|---|-|

In [9]:
def qiskit_params_map(circ):
    dag = circuit_to_dag(circ)
    param_list = [list(node.op.params[0].parameters)[0].name for node in dag.op_nodes() if node.op.params and isinstance(node.op.params[0], ParameterExpression)]
    return {k: v for v, k in enumerate(param_list)}

qiskit_param_map = qiskit_params_map(pcirc)

# Ensure the sorted names are correct
ordered_params = [param.name for param in pcirc.parameters]
assert sorted(qiskit_param_map.keys()) == ordered_params

param_map = { i:qiskit_param_map[param] for i, param in enumerate(ordered_params)}

In [10]:
stim_circ.define_parameter_map(param_map)

In [11]:
pcirc.parameters

ParameterView([Parameter(beta_0), Parameter(beta_1), Parameter(beta_2), Parameter(beta_3), Parameter(beta_4), Parameter(beta_5), Parameter(beta_6), Parameter(beta_7), Parameter(beta_8), Parameter(beta_9), Parameter(gamma_0), Parameter(gamma_1), Parameter(gamma_10), Parameter(gamma_11), Parameter(gamma_12), Parameter(gamma_13), Parameter(gamma_14), Parameter(gamma_15), Parameter(gamma_16), Parameter(gamma_17), Parameter(gamma_18), Parameter(gamma_19), Parameter(gamma_2), Parameter(gamma_20), Parameter(gamma_21), Parameter(gamma_22), Parameter(gamma_23), Parameter(gamma_24), Parameter(gamma_25), Parameter(gamma_26), Parameter(gamma_27), Parameter(gamma_28), Parameter(gamma_29), Parameter(gamma_3), Parameter(gamma_30), Parameter(gamma_31), Parameter(gamma_32), Parameter(gamma_33), Parameter(gamma_34), Parameter(gamma_35), Parameter(gamma_36), Parameter(gamma_37), Parameter(gamma_38), Parameter(gamma_39), Parameter(gamma_4), Parameter(gamma_40), Parameter(gamma_41), Parameter(gamma_42), Pa

In [12]:
# we can perform CAFQA by using the main optimization function "claptonize"

ks_best, _, energy_best = claptonize(
    reversed_paulis,
    coeffs,
    stim_circ,
    n_proc=4,           # total number of processes in parallel
    n_starts=4,         # number of random genetic algorithm starts in parallel
    n_rounds=1,         # number of budget rounds, if None it will terminate itself
    callback=print,     # callback for internal parameter (#iteration, energies, ks) processing
    budget=20           # budget per genetic algorithm instance
)

STARTING ROUND 0


started GA at id 1 with 1 procs
started GA at id None with 1 procs

started GA at id 2 with 1 procs


GA parameters used for this experiment:
  num_generations=20
  num_parents_mating=20
  population_size=100
  num_genes=65
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=10
started GA at id 3 with 1 procs


/Users/dbharadwaj/anaconda3/envs/qaoa/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


/Users/dbharadwaj/anaconda3/envs/qaoa/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")
/Users/dbharadwaj/anaconda3/envs/qaoa/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evoluti

GA parameters used for this experiment:
  num_generations=20
  num_parents_mating=20
  population_size=100
  num_genes=65
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=10GA parameters used for this experiment:
  num_generations=20
  num_parents_mating=20
  population_size=100
  num_genes=65
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=10



/Users/dbharadwaj/anaconda3/envs/qaoa/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


GA parameters used for this experiment:
  num_generations=20
  num_parents_mating=20
  population_size=100
  num_genes=65
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=10
[0, array([-242., -121., -121.,    0.]), array([3, 2, 0, 1, 1, 0, 3, 2, 0, 1, 3, 3, 0, 0, 3, 0, 0, 3, 1, 0, 3, 3,
       3, 0, 3, 2, 2, 0, 2, 0, 0, 2, 0, 2, 2, 1, 0, 1, 2, 0, 1, 1, 0, 1,
       0, 0, 2, 2, 0, 1, 1, 1, 3, 0, 3, 2, 2, 1, 0, 1, 2, 2, 3, 2, 2],
      dtype=object)]
[0, array([-242., -121., -121.,    0.]), array([3, 2, 0, 1, 1, 0, 3, 2, 0, 1, 3, 3, 0, 0, 3, 0, 0, 3, 1, 0, 3, 3,
       3, 0, 3, 2, 2, 0, 2, 0, 0, 2, 0, 2, 2, 1, 0, 1, 2, 0, 1, 1, 0, 1,
       0, 0, 2, 2, 0, 1, 1, 1, 3, 0, 3, 2, 2, 1, 0, 1, 2, 2, 3, 2, 2],
      dtype=object)]
[0, array([-242., -121., -121.,    0.]), array([3, 2, 0, 1, 1, 0, 3, 2, 0, 1, 3, 3, 0, 0, 3, 0, 0, 3, 1, 0, 3, 3,
       3, 0, 3, 

In [13]:
energy_best

np.float64(-495.0)

In [14]:
# Solve with classical Eigensolver for comparison
eigensolver = NumPyMinimumEigensolver()
exact_solution = eigensolver.compute_minimum_eigenvalue(cost_hamiltonian).eigenvalue.real
print("Exact Energy from Eigensolver:", exact_solution)

Exact Energy from Eigensolver: -1330.5


In [15]:
def cafqa_params_energy(circuit, hamiltonian, parameters):
    estimator = Estimator(mode=AerSimulator(method='statevector'))
    isa_hamiltonian = hamiltonian.apply_layout(circuit.layout)

    pub = (circuit, isa_hamiltonian, parameters)
    job = estimator.run([pub])

    results = job.result()[0]
    return results.data.evs

In [16]:
# Random Initalization 
random_angles = np.random.random(len(ks_best))
random_energies = [cafqa_params_energy(pcirc, cost_hamiltonian, random_angles) for _ in range(100)]
min_energy = min(random_energies)
print(f"Minimum Energy found with Random initialization over 100 runs: {min_energy}")

Minimum Energy found with Random initialization over 100 runs: 225.528076171875
